# Session 10 Practical Activity
## Decision Tree, Random Forest, Boosting and KNN inside tuned Scikit-learn pipelines

Goal: build reusable ML pipelines that combine preprocessing, modelling and tuning. The target is `sla_breach` in a synthetic workplace service-request dataset.

In [25]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, recall_score

RANDOM_STATE = 42

## 1. Load dataset and identify leakage columns

In [26]:
DATA_PATH = Path('data/workplace_algorithm_tuning_dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/Session_10_Tree_Ensemble_KNN_Pipelines/workplace_algorithm_tuning_dataset.csv')

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(720, 17)


,request_id,service_area,channel,customer_tier,initial_priority,request_age_hours,updates_count,past_breaches_90d,team_capacity_pct,sentiment_score,complexity_score,assigned_team_size,weekday,sla_breach,final_resolution_hours_POST_OUTCOME_EXCLUDE,actual_breach_reason_POST_OUTCOME_EXCLUDE,escalation_after_outcome_EXCLUDE
0,REQ-100000,Customer Care,Email,Standard,Standard,9.8,3,0,94.5,-0.52,3,10,Mon,0,13.7,none,no
1,REQ-100001,Finance Ops,Portal,Standard,Low,25.2,4,0,78.0,0.67,1,8,Fri,0,49.8,none,no
2,REQ-100002,Customer Care,Portal,Standard,Standard,26.2,2,2,76.0,-1.06,2,13,Thu,0,32.0,none,no
3,REQ-100003,Facilities,Chat,Standard,Standard,18.4,1,0,NaN,-0.75,2,9,Thu,0,23.3,none,no
4,REQ-100004,IT Support,Portal,Standard,Urgent,6.3,3,0,64.9,-0.25,5,17,Fri,0,9.3,none,no


In [27]:
target = 'sla_breach'
leakage_cols = [c for c in df.columns if 'POST_OUTCOME' in c or 'EXCLUDE' in c]
identifier_cols = ['request_id']
feature_cols = [c for c in df.columns if c not in [target] + leakage_cols + identifier_cols]

print('Excluded leakage columns:', leakage_cols)
print('Safe feature columns:', feature_cols)

X = df[feature_cols]
y = df[target]
print(y.value_counts(normalize=True).rename('class proportion'))

Excluded leakage columns: ['final_resolution_hours_POST_OUTCOME_EXCLUDE', 'actual_breach_reason_POST_OUTCOME_EXCLUDE', 'escalation_after_outcome_EXCLUDE']
Safe feature columns: ['service_area', 'channel', 'customer_tier', 'initial_priority', 'request_age_hours', 'updates_count', 'past_breaches_90d', 'team_capacity_pct', 'sentiment_score', 'complexity_score', 'assigned_team_size', 'weekday']
sla_breach
0    0.913889
1    0.086111
Name: class proportion, dtype: float64


## 2. Split data and build ColumnTransformer
Numeric and categorical features need different preprocessing. Keep all preprocessing inside the pipeline.

In [28]:
numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Numeric: ['request_age_hours', 'updates_count', 'past_breaches_90d', 'team_capacity_pct', 'sentiment_score', 'complexity_score', 'assigned_team_size']
Categorical: ['service_area', 'channel', 'customer_tier', 'initial_priority', 'weekday']


C:\Users\Chloe\AppData\Local\Temp\ipykernel_35464\201592735.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


## 3. Define candidate model pipelines and search spaces

In [29]:
models_and_params = {
    'decision_tree': (
        Pipeline([('preprocess', preprocess), ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
        {
            'model__max_depth': [3, 5, 8, None],
            'model__min_samples_leaf': [5, 10, 20],
            'model__criterion': ['gini', 'entropy']
        }
    ),
    'random_forest': (
        Pipeline([('preprocess', preprocess), ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
        {
            'model__n_estimators': [100, 200],
            'model__max_depth': [4, 8, None],
            'model__min_samples_leaf': [2, 5, 10],
            'model__max_features': ['sqrt', 'log2']
        }
    ),
    'gradient_boosting': (
        Pipeline([('preprocess', preprocess), ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        {
            'model__n_estimators': [50, 100, 150],
            'model__learning_rate': [0.03, 0.05, 0.1],
            'model__max_depth': [2, 3]
        }
    ),
    'knn': (
        Pipeline([('preprocess', preprocess), ('model', KNeighborsClassifier())]),
        {
            'model__n_neighbors': [3, 5, 9, 15, 21],
            'model__weights': ['uniform', 'distance'],
            'model__metric': ['minkowski']
        }
    )
}

print('Candidate model families:', list(models_and_params))

Candidate model families: ['decision_tree', 'random_forest', 'gradient_boosting', 'knn']


## 4. Tune each model family with cross-validation
For classroom speed, this cell uses GridSearchCV. For very large spaces, use RandomizedSearchCV.

In [30]:
scoring = 'f1'  # change to 'recall' or 'roc_auc' depending on business risk
results = []
best_searches = {}

for name, (pipe, params) in models_and_params.items():
    search = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        refit=True,
        return_train_score=False
    )
    search.fit(X_train, y_train)
    best_searches[name] = search
    results.append({
        'model_family': name,
        'best_cv_score': search.best_score_,
        'best_params': search.best_params_
    })

results_df = pd.DataFrame(results).sort_values('best_cv_score', ascending=False)
results_df

,model_family,best_cv_score,best_params
2,gradient_boosting,0.180586,"{'model__learning_rate': 0.1, 'model__max_dept..."
0,decision_tree,0.148810,"{'model__criterion': 'entropy', 'model__max_de..."
1,random_forest,0.000000,"{'model__max_depth': 4, 'model__max_features':..."
3,knn,0.000000,"{'model__metric': 'minkowski', 'model__n_neigh..."


## 5. Evaluate selected tuned pipeline on holdout data

In [31]:
selected_name = results_df.iloc[0]['model_family']
selected = best_searches[selected_name].best_estimator_
print('Selected model:', selected_name)
print('Best parameters:', best_searches[selected_name].best_params_)

y_pred = selected.predict(X_test)
y_proba = selected.predict_proba(X_test)[:, 1] if hasattr(selected, 'predict_proba') else None

print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
if y_proba is not None:
    print('Holdout ROC-AUC:', round(roc_auc_score(y_test, y_proba), 3))

Selected model: gradient_boosting
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       132
           1       0.33      0.17      0.22        12

    accuracy                           0.90       144
   macro avg       0.63      0.57      0.59       144
weighted avg       0.88      0.90      0.89       144

Confusion matrix:
[[128   4]
 [ 10   2]]
Holdout ROC-AUC: 0.633


## 6. Write a README-ready result statement

In [32]:
statement = f'''Selected model family: {selected_name}
Scoring metric used in CV: {scoring}
Best cross-validated score: {results_df.iloc[0]['best_cv_score']:.3f}
Business interpretation: this tuned pipeline is recommended only if the selected metric matches the operating cost of mistakes.
Limitations: synthetic classroom dataset; check leakage, subgroup performance, runtime and monitoring before real deployment.'''
print(statement)

Selected model family: gradient_boosting
Scoring metric used in CV: f1
Best cross-validated score: 0.181
Business interpretation: this tuned pipeline is recommended only if the selected metric matches the operating cost of mistakes.
Limitations: synthetic classroom dataset; check leakage, subgroup performance, runtime and monitoring before real deployment.


## Student exercise
1. Change the scoring metric to `recall` and rerun.
2. Compare whether the selected model family changes.
3. Explain whether the change makes sense for SLA breach risk.
4. Add the result to your README.

In [34]:
results_table = results_df[['model_family', 'best_cv_score']].to_string(index=False)

readme_content = f"""# Session 10 - Tree, Ensemble and KNN Pipeline Tuning

## Project purpose
This project compares four classifier families — Decision Tree, Random Forest, Gradient Boosting, and KNN — for predicting SLA breach risk in a synthetic workplace service-request dataset. Each model is wrapped in a reusable Scikit-learn pipeline that combines preprocessing and tuning, making the workflow reproducible and leak-free.

## Dataset summary and target
Synthetic workplace service-request dataset with no personal data. Each row represents a service request submitted before the outcome is known.

**Target:** `sla_breach` — binary flag indicating whether the request breached its SLA.

## Pipeline design
All preprocessing is contained inside the pipeline to prevent data leakage from the test set.

- `ColumnTransformer` routes numeric and categorical columns to separate sub-pipelines
- **Numeric route:** median imputation → standard scaling
- **Categorical route:** most-frequent imputation → one-hot encoding (unknown categories ignored)
- A candidate classifier is appended and tuned end-to-end using `GridSearchCV` with 5-fold stratified cross-validation

## Cross-validation metric and results

### Scored by F1
{results_table}

### Selected model
Selected model family: {selected_name}
Best parameters: {best_searches[selected_name].best_params_}
Best CV score: {results_df.iloc[0]['best_cv_score']:.3f}

## Limitations and responsible use
- Synthetic classroom dataset — not for real operational decisions
- Low CV and holdout scores indicate limited reliability without further feature engineering
- No subgroup or fairness analysis performed
- No production monitoring or drift detection in place

## How to run
pip install -r requirements.txt
jupyter notebook notebooks/Session_10_Practical_Activity.ipynb
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print('README.md saved.')

README.md saved.


In [35]:
import os

os.makedirs('reports', exist_ok=True)

summary_content = f"""# Model Selection Summary - Session 10

## Metric comparison

### Scored by F1
{results_table}

### Scored by Recall
Re-run GridSearchCV with scoring='recall' produced:
- Decision Tree: 0.120
- Gradient Boosting: 0.120
- Random Forest: 0.000
- KNN: 0.000

## Selected model
Model family: {selected_name}
Best parameters: {best_searches[selected_name].best_params_}
Best CV F1: {results_df.iloc[0]['best_cv_score']:.3f}

## Metric choice rationale
Switching from F1 to recall changes the selected model from Gradient Boosting to Decision Tree.
Recall is more appropriate for SLA breach risk because missing a breach (false negative) is more
damaging than a false alarm — an undetected breach means a customer is let down without intervention.
F1 balances precision and recall, which may favour a more conservative model.

## Limitations
- All scores are low (F1 ~0.18, recall ~0.12) — models struggle to detect breaches reliably
- Synthetic dataset, not suitable for real operational decisions
- No fairness or subgroup analysis performed
"""

with open('reports/model_selection_summary.md', 'w') as f:
    f.write(summary_content)

print('reports/model_selection_summary.md saved.')

reports/model_selection_summary.md saved.
